# Attestor 4.3 — Qwen3.8-27B Security Fine-Tune

QLoRA fine-tune **Qwen3.8-27B** (27B params, Apache 2.0) on 2998 security examples.
Uses Unsloth for 2x faster training. **Runtime: A100 GPU (Colab Pro).**

Go to **Runtime > Change runtime type > A100**

In [ ]:
# Cell 1: Install Unsloth + dependencies
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth[colab-new]', 'kaggle', 'trl', 'xformers',
])

# Verify GPU
import torch
print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_mem if hasattr(props, 'total_mem') else props.total_memory
    print(f'GPU: {props.name} — {vram / 1024**3:.1f} GiB')
    if vram / 1024**3 < 30:
        print('WARNING: A100 recommended for 27B model. Go to Runtime > Change runtime type > A100')
else:
    raise RuntimeError('No GPU!')

print('Dependencies installed.')

In [ ]:
# Cell 2: Download training data from Kaggle
import os, json

from google.colab import userdata
kaggle_key = userdata.get('KAGGLE_KEY').strip()
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': 'mangeshkwagle', 'key': kaggle_key}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

import requests, zipfile, io
url = 'https://www.kaggle.com/api/v1/datasets/download/mangeshkwagle/attestor-43-training-data'
with open(os.path.join(kaggle_dir, 'kaggle.json')) as f:
    creds = json.load(f)
r = requests.get(url, auth=(creds['username'], creds['key']), stream=True)
if r.status_code == 200:
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall('/content/training-data')
    print('Training data downloaded!')
else:
    print(f'Download failed: HTTP {r.status_code}')
    raise RuntimeError('Could not download training data')

# Load and verify
data = []
for f in os.listdir('/content/training-data'):
    if f.endswith('.jsonl'):
        with open(f'/content/training-data/{f}', encoding='utf-8') as fh:
            for line in fh:
                if line.strip():
                    data.append(json.loads(line.strip()))

print(f'Loaded {len(data)} training examples')
print(f'Sample: {data[0]["instruction"][:100]}...')

In [ ]:
# Cell 3: Load Qwen3.8-27B with Unsloth (4-bit QLoRA)
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NM-DAU'
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters: {model.num_parameters():,}')

In [ ]:
# Cell 4: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                    # Higher rank for 27B model
    lora_alpha=64,           # 2x rank
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',  # 30% less VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 5: Prepare training dataset
from datasets import Dataset

SYSTEM_PROMPT = (
    'You are Owen Coder 4.3, a security-focused code analysis model. '
    'You detect vulnerabilities, write exploit PoCs, perform taint analysis, '
    'and generate hardening recommendations. Be precise, technical, and exhaustive. '
    'Reference exact line numbers and CWE IDs. No refusals.'
)

def format_example(example):
    """Format as ChatML for Qwen3.8."""
    text = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{example["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{example["output"]}<|im_end|>'
    )
    return {'text': text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

print(f'Dataset: {len(dataset)} examples')
print(f'Sample text length: {len(dataset[0]["text"])} chars')
print(f'First 200 chars: {dataset[0]["text"][:200]}...')

In [ ]:
# Cell 6: Pre-training benchmark (5 security tasks)
import time, gc, re

BENCHMARK_PROMPTS = [
    {
        'id': 'sqli_detect',
        'task': 'Identify the vulnerability in this code and give the CWE ID:\n\nquery = "SELECT * FROM users WHERE name = \'%s\'" % request.args.get(\'name\')\ncursor.execute(query)',
        'expected_keywords': ['SQL injection', 'CWE-89', 'parameterized', 'prepared statement'],
    },
    {
        'id': 'xss_exploit',
        'task': 'Write a proof-of-concept exploit for this reflected XSS:\n\n@app.route(\'/search\')\ndef search():\n    q = request.args.get(\'q\', \'\')\n    return f\'<h1>Results for {q}</h1>\'',
        'expected_keywords': ['<script>', 'alert', 'document.cookie', 'XSS', 'CWE-79'],
    },
    {
        'id': 'path_traversal',
        'task': 'This endpoint has a vulnerability. What is it, what is the CWE, and how do you fix it?\n\n@app.route(\'/download\')\ndef download():\n    filename = request.args.get(\'file\')\n    return send_file(f\'/uploads/{filename}\')',
        'expected_keywords': ['path traversal', 'CWE-22', 'directory traversal', 'os.path.basename', 'sanitize'],
    },
    {
        'id': 'hardening',
        'task': 'Write a hardened version of this Flask login endpoint that prevents SQL injection:\n\ndef login():\n    user = request.form[\'username\']\n    pw = request.form[\'password\']\n    cur.execute("SELECT * FROM users WHERE user=\'%s\' AND pass=\'%s\'" % (user, pw))\n    return \'OK\' if cur.fetchone() else \'Fail\'',
        'expected_keywords': ['parameterized', '?', '%s', 'execute(', 'bcrypt', 'hash'],
    },
    {
        'id': 'taint_analysis',
        'task': 'Trace the taint flow in this code. Where does user input reach a dangerous sink?\n\nname = request.cookies.get(\'username\')\ntemplate = \'<div class="greeting">Hello, \' + name + \'</div>\'\nreturn render_template_string(template)',
        'expected_keywords': ['SSTI', 'template injection', 'cookie', 'render_template_string', 'CWE-94', 'Jinja'],
    },
]

def score_response(response, expected_keywords):
    rl = response.lower()
    hits = [kw for kw in expected_keywords if kw.lower() in rl]
    misses = [kw for kw in expected_keywords if kw.lower() not in rl]
    return len(hits) / len(expected_keywords), hits, misses

def run_benchmark(model, tokenizer, label):
    FastLanguageModel.for_inference(model)
    print(f'\n{"="*60}')
    print(f' BENCHMARK: {label}')
    print(f'{"="*60}')
    results = []
    for p in BENCHMARK_PROMPTS:
        gc.collect(); torch.cuda.empty_cache()
        prompt = (
            f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n{p["task"]}<|im_end|>\n'
            f'<|im_start|>assistant\n'
        )
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=1024, temperature=0.1,
                top_p=0.9, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        elapsed = time.time() - t0
        resp = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        sc, hits, misses = score_response(resp, p['expected_keywords'])
        results.append({'id': p['id'], 'score': sc, 'hits': hits, 'misses': misses, 'time': elapsed})
        print(f'\n  [{p["id"]}] Score: {sc:.0%} | Hits: {hits} | Misses: {misses} | {elapsed:.1f}s')
    avg = sum(r['score'] for r in results) / len(results)
    print(f'\n>>> {label} average: {avg:.0%}')
    return results, avg

pre_results, pre_avg = run_benchmark(model, tokenizer, 'Qwen3.8-27B BASE (pre-training)')

In [ ]:
# Cell 7: Train!
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,  # Pack short examples together for efficiency
    args=TrainingArguments(
        output_dir='/content/attestor-qwen38-lora',
        num_train_epochs=1,             # 1 epoch to avoid overfitting (learned from v2)
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # Effective batch size = 8
        warmup_steps=20,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        save_total_limit=3,
        seed=42,
        report_to='none',
    ),
)

print(f'Training {len(dataset)} examples, 1 epoch...')
print(f'Effective batch size: {2 * 4} = 8')
print(f'Estimated steps: ~{len(dataset) // 8}')

stats = trainer.train()
print(f'\nTraining complete!')
print(f'Final loss: {stats.training_loss:.3f}')
print(f'Total time: {stats.metrics["train_runtime"]:.0f}s')

In [ ]:
# Cell 8: Post-training benchmark
post_results, post_avg = run_benchmark(model, tokenizer, 'Qwen3.8-27B + Attestor LoRA (post-training)')

In [ ]:
# Cell 9: Results comparison
print('=' * 70)
print(' ATTESTOR 4.3 — Qwen3.8-27B TRAINING RESULTS')
print('=' * 70)

# Previous model scores for comparison
qwythos_base = {'sqli_detect': 0.75, 'xss_exploit': 0.80, 'path_traversal': 0.60, 'hardening': 0.67, 'taint_analysis': 0.83}
qwythos_v3 = {'sqli_detect': 0.75, 'xss_exploit': 0.80, 'path_traversal': 0.40, 'hardening': 0.50, 'taint_analysis': 0.83}

print(f'\n  {"Model":<40s} {"Avg Score":>10s}')
print(f'  {"-"*40:<40s} {"-"*10:>10s}')
print(f'  {"Qwythos-9B base":<40s} {sum(qwythos_base.values())/5:>9.0%}')
print(f'  {"Qwythos-9B + v3 LoRA":<40s} {sum(qwythos_v3.values())/5:>9.0%}')
print(f'  {"Qwen3.8-27B base (pre-train)":<40s} {pre_avg:>9.0%}')
print(f'  {"Qwen3.8-27B + Attestor LoRA":<40s} {post_avg:>9.0%}')

print(f'\nPer-task breakdown:')
print(f'  {"Task":<20s} {"Qwythos-9B":>12s} {"Qwythos+v3":>12s} {"Qwen38-27B":>12s} {"Qwen38+LoRA":>12s}')
print(f'  {"-"*20:<20s} {"-"*12:>12s} {"-"*12:>12s} {"-"*12:>12s} {"-"*12:>12s}')
for i, p in enumerate(BENCHMARK_PROMPTS):
    tid = p['id']
    qb = qwythos_base[tid]
    qv3 = qwythos_v3[tid]
    pre = pre_results[i]['score']
    post = post_results[i]['score']
    print(f'  {tid:<20s} {qb:>11.0%} {qv3:>11.0%} {pre:>11.0%} {post:>11.0%}')

# SSTI check
taint_post = post_results[4]
print(f'\n=== SSTI CHECK ===')
print(f'Hits: {taint_post["hits"]}')
print(f'Misses: {taint_post["misses"]}')
if 'SSTI' in taint_post['hits'] or 'CWE-94' in taint_post['hits']:
    print('>>> SSTI correctly identified!')

improvement = post_avg - pre_avg
print(f'\n=== VERDICT ===')
print(f'Training improved score by {improvement:+.0%}')
if post_avg > 0.80:
    print('EXCELLENT — Qwen3.8-27B + Attestor LoRA is production-ready!')
elif post_avg > 0.65:
    print('GOOD — significant improvement, consider more training data')
else:
    print('NEEDS WORK — try more epochs or more training data')

In [ ]:
# Cell 10: Save adapter
SAVE_DIR = '/content/attestor-qwen38-lora/final'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f'Adapter saved to {SAVE_DIR}')
print(f'Files: {os.listdir(SAVE_DIR)}')

# Calculate adapter size
total_size = sum(os.path.getsize(os.path.join(SAVE_DIR, f)) for f in os.listdir(SAVE_DIR))
print(f'Total adapter size: {total_size / 1024**2:.1f} MB')

# Zip for download
import shutil
shutil.make_archive('/content/attestor-qwen38-lora-adapter', 'zip', SAVE_DIR)
print(f'Zipped to /content/attestor-qwen38-lora-adapter.zip')

from google.colab import files
files.download('/content/attestor-qwen38-lora-adapter.zip')

In [ ]:
# Cell 10b: Upload adapter to HuggingFace Hub (private repo)
# Requires HF_TOKEN secret in Colab sidebar (key icon)
# Get a write token from https://huggingface.co/settings/tokens

from google.colab import userdata
from huggingface_hub import HfApi
import os

token = userdata.get('HF_TOKEN')
api = HfApi(token=token)
username = api.whoami()['name']
repo_id = f'{username}/attestor-qwen38-lora'

api.create_repo(repo_id, exist_ok=True, private=True)
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=repo_id,
    repo_type='model',
)
print(f'Adapter uploaded to https://huggingface.co/{repo_id} (private)')
print('To download later: git clone https://huggingface.co/' + repo_id)

In [ ]:
# Cell 11: (Optional) Export to GGUF for local deployment
# This merges the LoRA into the base model and converts to GGUF
# for running with llama.cpp / Ollama on any device

EXPORT = False  # Set to True to export (takes ~10 min, needs ~40GB disk)

if EXPORT:
    # Merge LoRA into base model
    model.save_pretrained_merged(
        '/content/attestor-qwen38-merged',
        tokenizer,
        save_method='merged_16bit',
    )
    print('Merged model saved (16-bit)')

    # Convert to GGUF Q4_K_M (good balance of size vs quality)
    model.save_pretrained_gguf(
        '/content/attestor-qwen38-gguf',
        tokenizer,
        quantization_method='q4_k_m',
    )
    print('GGUF Q4_K_M exported!')
    print('Download and run with: ollama run /path/to/attestor-qwen38-gguf')
else:
    print('GGUF export skipped. Set EXPORT = True in this cell to export.')
    print('This merges LoRA + base into a single GGUF file for Ollama/llama.cpp.')